# Analysis of Arabic MCQ Surveys

## 1. Setup and Imports

In [1]:
import os
import random
import re
import pandas as pd
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
from matplotlib import font_manager
import seaborn as sns
import missingno as msno
import numpy as np
from dotenv import load_dotenv
from Kobo.kobo_toolbox_client import KoboToolboxClient
from Kobo.kobo_form_builder import KoboFormBuilder
from Kobo.kobo_form_uploader import KoboFormUploader
import featuretools as ft

c:\Users\eyasw\OneDrive\Desktop\Gra\analytics-of-arabic-mcq-surveys\.venv\Lib\site-packages\woodwork\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1.1 Configure Environment
Load environment variables from a `.env` file. This is where you should store your API token securely.

In [2]:
load_dotenv()

True

### 1.2 Configure Arabic Font for Plots

In [3]:
font_path = 'C:/Windows/Fonts/arial.ttf'  # Change this to a font that supports Arabic
if os.path.exists(font_path):
    prop = font_manager.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
else:
    print(f'Font not found at {font_path}. Please update the path.')

## 2. KoboToolbox Integration

### 2.1 Initialize KoboToolbox Client
The client is initialized using the URL and API token from your environment variables.

In [4]:
KOBO_API_URL = os.getenv("KOBO_API_URL", "https://kf.kobotoolbox.org/")
KOBO_API_TOKEN = os.getenv("KOBO_API_TOKEN")

if not KOBO_API_TOKEN:
    raise ValueError("KOBO_API_TOKEN not found in environment variables. Please set it in your .env file.")

kobo_client = KoboToolboxClient(
	url=KOBO_API_URL,
	token=KOBO_API_TOKEN
)

### 2.2 Create a New Survey Form

In [5]:
builder = KoboFormBuilder('استبيان رضا المستفيدين')

# Section 1: Demographic Questions
builder.add_select_question(name='gender', label='ما هو جنسك؟', options=['ذكر', 'أنثى'], required=True)
builder.add_select_question(name='age_group', label='ما هو فئتك العمرية؟', options=['أقل من 18', '18-24', '25-34', '35-44', '45 فأكثر'], required=True)
builder.add_select_question(name='governorate', label='في أي محافظة تعيش؟', options=['دمشق', 'ريف دمشق', 'حلب', 'حمص', 'حماة', 'اللاذقية', 'طرطوس', 'إدلب', 'درعا', 'القنيطرة', 'دير الزور', 'الحسكة', 'الرقة', 'السويداء'], required=True)
builder.add_select_question(name='education_level', label='ما هو مستواك التعليمي؟', options=['ثانوي أو أقل', 'دبلوم', 'بكالوريوس', 'ماجستير', 'دكتوراه'], required=True)
builder.add_select_question(name='employment_status', label='ما هو وضعك الوظيفي؟', options=['طالب', 'موظف', 'باحث عن عمل', 'عمل حر', 'أخرى'], required=True)
builder.add_select_question(name='specialization', label='ما هو تخصصك الأكاديمي؟', options=['هندسة حاسوب', 'ذكاء اصطناعي', 'إدارة أعمال', 'اقتصاد', 'علوم اجتماعية', 'تربية', 'أخرى'], required=False)

# Section 2: Satisfaction Measurement
builder.add_select_question(name='satisfaction_overall', label='كيف تقيم تجربتك العامة مع الخدمات التي حصلت عليها؟', options=['راضٍ جدًا', 'راضٍ', 'محايد', 'غير راضٍ', 'غير راضٍ أبدًا'], required=True)
builder.add_select_question(name='satisfaction_access', label='ما مدى سهولة الوصول إلى الخدمات المقدمة؟', options=['سهل جدًا', 'سهل', 'متوسط', 'صعب', 'صعب جدًا'], required=True)
builder.add_select_question(name='satisfaction_support', label='ما مدى جودة الدعم الذي حصلت عليه أثناء استخدام الخدمة؟', options=['ممتاز', 'جيد جدًا', 'جيد', 'ضعيف', 'سيء جدًا'], required=True)
builder.add_select_question(name='suggestions', label='هل لديك اقتراحات لتحسين الخدمات؟', options=['نعم، لدي اقتراح', 'لا يوجد لدي اقتراح'], required=False)

form_json = builder.to_json()
print("Form JSON created successfully.")

Form JSON created successfully.


### 2.3 Upload and Deploy the Form
This step will create and deploy the form on KoboToolbox. You only need to run this once.

In [6]:
uploader = KoboFormUploader(api_url=KOBO_API_URL, token=KOBO_API_TOKEN)

# try:
#     # 1) Create the form
#     result = uploader.create_form(form_json, builder.title)
#     form_uid = result.get('uid') or result.get('id')
#     print(f"Form created with UID: {form_uid}")

#     # 2) Deploy the form
#     deployment = uploader.deploy_form(form_uid)
#     print(f"Form deployed successfully. Identifier: {deployment.get('identifier')}")
# except Exception as e:
#     print(f"An error occurred during form upload/deployment: {e}")

## 3. Data Retrieval and Analysis

### 3.1 Fetch Submissions from KoboToolbox

In [7]:
try:
    forms = kobo_client.get_all_forms()
    if forms:
        first_form_uid = forms[0].uid
        print(f"Fetching submissions for form: {forms[0].name} ({first_form_uid})")
        submissions_df = kobo_client.get_submissions(first_form_uid)
        display(submissions_df.head())
    else:
        print("No forms found.")
except Exception as e:
    print(f'\nError occurred while fetching submissions: {e}')

Fetching submissions for form: استبيان رضا المستفيدين (aKmraftnzKaSRwfcDqrN27)


,_id,formhub/uuid,ما هو جنسك؟,ما هو فئتك العمرية؟,في أي محافظة تعيش؟,ما هو مستواك التعليمي؟,ما هو وضعك الوظيفي؟,ما هو تخصصك الأكاديمي؟,كيف تقيم تجربتك العامة مع الخدمات التي حصلت عليها؟,ما مدى سهولة الوصول إلى الخدمات المقدمة؟,...,_uuid,meta/rootUuid,_attachments,_status,_geolocation,_submission_time,_tags,_notes,_validation_status,_submitted_by
0,513119512,1e91a958e62d444faac93333f9fb7fc5,NaN,18-24,دمشق,بكالوريوس,طالب,هندسة حاسوب,راضٍ,متوسط,...,00239809-cd17-4a06-ac10-fb5fc875aaa6,uuid:00239809-cd17-4a06-ac10-fb5fc875aaa6,[],submitted_via_web,"[None, None]",2025-07-09T05:52:30,[],[],{},None
1,513119829,1e91a958e62d444faac93333f9fb7fc5,NaN,18-24,دمشق,بكالوريوس,طالب,هندسة حاسوب,راضٍ,متوسط,...,72475b88-fbad-41fb-97db-ac859162db1b,uuid:72475b88-fbad-41fb-97db-ac859162db1b,[],submitted_via_web,"[None, None]",2025-07-09T05:53:11,[],[],{},None
2,513120074,1e91a958e62d444faac93333f9fb7fc5,NaN,25-34,حمص,ماجستير,موظف,اقتصاد,محايد,صعب,...,c80a7d33-629a-481b-a9ca-0a4a2be57a64,uuid:c80a7d33-629a-481b-a9ca-0a4a2be57a64,[],submitted_via_web,"[None, None]",2025-07-09T05:53:45,[],[],{},None
3,513120231,1e91a958e62d444faac93333f9fb7fc5,NaN,18-24,دمشق,بكالوريوس,موظف,ذكاء اصطناعي,غير راضٍ,متوسط,...,fe4810d4-5b51-4aa2-b115-fd744b61a42f,uuid:fe4810d4-5b51-4aa2-b115-fd744b61a42f,[],submitted_via_web,"[None, None]",2025-07-09T05:54:09,[],[],{},None
4,513705945,1e91a958e62d444faac93333f9fb7fc5,NaN,NaN,دير الزور,ماجستير,موظف,ذكاء اصطناعي,محايد,سهل,...,838ad313-fbfc-4a84-bc28-6f221cd4ad98,uuid:838ad313-fbfc-4a84-bc28-6f221cd4ad98,[],submitted_via_web,"[None, None]",2025-07-09T18:37:30,[],[],{},None


### 3.2 Map Coded Answers to Labels

In [ ]:
def map_answers_to_labels(kobo_client : KoboToolboxClient, form_uid : str, submissions_df : pd.DataFrame):
    """Maps coded answers in a DataFrame to their human-readable labels."""
    try:
        form_details = kobo_client.get_form_details(form_uid)
        
        # Create a mapping from choice list names to their options
        choice_map = {}
        for choice in form_details.get('choices', []):
            list_name = choice.get('list_name')
            if list_name not in choice_map:
                choice_map[list_name] = {}
            choice_map[list_name][choice.get('name')] = choice.get('label', [None])[0]

        # Create a mapping from question names to their choice maps
        question_map = {}
        for question in form_details.get('survey', []):
            if question.get('type') in ['select_one', 'select_multiple']:
                list_name = question.get('select_from_list_name')
                if list_name in choice_map:
                    question_map[question.get('name')] = choice_map[list_name]

        # Apply the mapping to the DataFrame
        df_labeled = submissions_df.copy()
        for col, mapping in question_map.items():
            if col in df_labeled.columns:
                df_labeled[col] = df_labeled[col].map(mapping)
        
        return df_labeled

    except Exception as e:
        print(f"An error occurred while mapping answers: {e}")
        return submissions_df # Return original df on error

if 'submissions_df' in locals() and not submissions_df.empty:
    submissions_labeled_df = map_answers_to_labels(kobo_client, first_form_uid, submissions_df)
    print("DataFrame with human-readable labels:")
    display(submissions_labeled_df.head())
else:
    print("Submissions DataFrame is not available or is empty.")

### 3.3 Exploratory Data Analysis (EDA)

In [ ]:
if 'submissions_labeled_df' in locals() and not submissions_labeled_df.empty:
    print("Dataset Shape:", submissions_labeled_df.shape)
    print("\nMissing Values:")
    msno.matrix(submissions_labeled_df)
    plt.show()
else:
    print("Labeled submissions DataFrame is not available or is empty.")

In [ ]:
def plot_arabic_bar(data, column, title):
    """Helper function to plot bar charts with Arabic labels."""
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(y=data[column], order=data[column].value_counts().index)
    
    # Reshape and display Arabic labels correctly
    ax.set_yticklabels([get_display(arabic_reshaper.reshape(label.get_text())) for label in ax.get_yticklabels()])
    
    plt.title(get_display(arabic_reshaper.reshape(title)))
    plt.xlabel(get_display(arabic_reshaper.reshape('العدد')))
    plt.ylabel('')
    plt.tight_layout()
    plt.show()

if 'submissions_labeled_df' in locals() and not submissions_labeled_df.empty:
    plot_arabic_bar(submissions_labeled_df, 'gender', 'توزيع المشاركين حسب الجنس')
    plot_arabic_bar(submissions_labeled_df, 'age_group', 'توزيع المشاركين حسب الفئة العمرية')
    plot_arabic_bar(submissions_labeled_df, 'governorate', 'توزيع المشاركين حسب المحافظة')
else:
    print("Labeled submissions DataFrame is not available or is empty.")

## 4. Automated Feature Engineering

In [ ]:
if 'submissions_labeled_df' in locals() and not submissions_labeled_df.empty:
    # Create an EntitySet
    es = ft.EntitySet(id="survey_data")

    # Add the submissions dataframe to the EntitySet
    es = es.add_dataframe(
        dataframe_name="submissions", 
        dataframe=submissions_labeled_df, 
        index="_index",
        time_index="_submission_time"
    )

    # Run Deep Feature Synthesis (DFS)
    feature_matrix, feature_defs = ft.dfs(
        entityset=es,
        target_dataframe_name="submissions",
        agg_primitives=["count", "sum", "mean", "std", "skew", "min", "max", "mode"],
        trans_primitives=["day", "year", "month", "weekday", "is_weekend"],
        max_depth=2
    )

    print("Feature matrix shape:", feature_matrix.shape)
    display(feature_matrix.head())

else:
    print("Labeled submissions DataFrame is not available or is empty.")